# 08 — ML Transformers

Feature-engineering transformers: `VectorAssembler`, `StringIndexer`, `OneHotEncoder`, `StandardScaler`, `QuantileDiscretizer`. Each follows a `fit`/`transform` pattern.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Synthetic data

In [ ]:
import pandas as pd
from irispark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, QuantileDiscretizer

pdf = pd.DataFrame({
    "idade": [25, 30, 35, 40, 45, 50],
    "renda": [3000, 5000, 7000, 9000, 11000, 13000],
    "cidade": ["SP", "RJ", "MG", "SP", "RJ", "MG"],
})
df = session.createDataFrame(pdf)
df.show()

## 2. `StringIndexer`

Encode a categorical column as numeric indices.

In [ ]:
si = StringIndexer(inputCol="cidade", outputCol="cidade_idx")
si.fit(df).transform(df).show()

## 3. `OneHotEncoder`

One-hot encode a categorical column.

In [ ]:
si = StringIndexer(inputCol="cidade", outputCol="cidade_idx")
df_idx = si.fit(df).transform(df)
ohe = OneHotEncoder(inputCol="cidade_idx", outputCol="cidade_ohe")
ohe.fit(df_idx).transform(df_idx).show()

## 4. `StandardScaler`

Standardize a numeric column (zero mean, unit variance).

In [ ]:
ss = StandardScaler(inputCol="renda", outputCol="renda_std")
ss.fit(df).transform(df).show()

## 5. `QuantileDiscretizer`

Bin a numeric column into quantile buckets.

In [ ]:
qd = QuantileDiscretizer(inputCol="renda", outputCol="renda_bin", numBuckets=3)
qd.fit(df).transform(df).show()

## 6. `VectorAssembler`

Assemble numeric columns into a feature vector.

In [ ]:
va = VectorAssembler(inputCols=["idade", "renda"], outputCol="features")
va.transform(df).show()

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")